# 手动实现一个PPO 训练CartPole

这份代码是没有reference model的， 他是2017提出的PPO算法， 因此没有reference model，后面的RLHF有

In [2]:
import argparse
import os
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

KeyboardInterrupt: 

1. 由于Cartpole是一个让杆子不要乱动的游戏，因此actor只有一个左移，和右移动，所以下面act_dim = 2 ， 所以这里action的结果是0或者1
2. 

## 定义一个Actor 和 Critic 函数

In [ ]:
class ActorCritic(nn.Module):
    """
    独立 Actor-Critic 网络（与 SB3 MlpPolicy 对齐）：
    1. Actor 和 Critic 使用各自的隐藏层，避免梯度冲突
    2. 正交初始化：actor 输出层 gain=0.01 保证初始策略接近均匀分布
    """

    def __init__(self, obs_dim=4, act_dim=2, hidden=64):
        super().__init__()
        self.actor = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, act_dim),
        )
        self.critic = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, 1),
        )
        self._init_weights()

    def _init_weights(self):
        """正交初始化，与 SB3 默认一致"""
        for module in self.actor:
            if isinstance(module, nn.Linear):
                nn.init.orthogonal_(module.weight, gain=np.sqrt(2))
                nn.init.constant_(module.bias, 0)
        for module in self.critic:
            if isinstance(module, nn.Linear):
                nn.init.orthogonal_(module.weight, gain=np.sqrt(2))
                nn.init.constant_(module.bias, 0)
        # actor 初始化， 最后一层的参数矩阵乘以0.01，让开始的结果概率近似均衡分布
        nn.init.orthogonal_(self.actor[-1].weight, gain=0.01)
        nn.init.constant_(self.actor[-1].bias, 0)
        # critic 输出层 gain=1
        nn.init.orthogonal_(self.critic[-1].weight, gain=1.0)
        nn.init.constant_(self.critic[-1].bias, 0)

    def forward(self, x):
        logits = self.actor(x)
        value = self.critic(x)
        return logits, value.squeeze(-1)

    def get_action(self, obs, deterministic=False):
        """这里action是一个向量, 具体是0或者1，表示是采取向左还是想右"""
        # 这里logits是未经过softmax的原始输出
        
        # Actor给出的logits表示每个动作的概率，Critic给出的value表示当前状态的价值估计
        logits, value = self.forward(obs)
        
        # 会把logits转换成总和为1的概率分布，并且根据这个分布采样动作
        dist = torch.distributions.Categorical(logits=logits)
        if deterministic:
            # 选择概率最大的动作
            action = logits.argmax(dim=-1)
        else:
            # 根据概率分布采样动作
            action = dist.sample()
        log_prob = dist.log_prob(action)
        return action, log_prob, value

# 收集轨迹

这里obs代表当前环境的函数
1. action, log_prob, value = model.get_action(obs_tensor) 根据当前环境obs_tensor, 获取actor的行为（action）， 以及该action对应的概率的ln结果(log_prob)， 以及critic预测未来的预期是Value


2. env.step(action.item()):
   next_obs, reward, terminated, truncated, _ = env.step(action.item())
   使用action来拿到下一步的状态和当前这一步所获得的奖励reward

3.  transitions.append 把记录记录下来， 然后如果不是自然结束（terminated），那么就把他的boot_strap给记录下来，因为需要这个来预测他的奖励
    1.  "next_obs": next_obs if truncated and not terminated else None,
    2.  _, _, bootstrap_value = model.get_action(torch.FloatTensor(obs))
   

In [ ]:
# ==========================================
# 第二部分：收集轨迹（Rollout）
# ==========================================
def collect_rollout(model, env, num_steps=2048):
    """
    收集轨迹，正确处理 terminated vs truncated：
    - terminated（杆子倒了）：V(s')=0
    - truncated（达到步数上限）：V(s')需要 bootstrap
    - rollout 末尾未结束：需要 bootstrap
    """
    # obs是当前环境状态
    obs, _ = env.reset()
    transitions = []

    for _ in range(num_steps):
        obs_tensor = torch.FloatTensor(obs)
        with torch.no_grad():
            action, log_prob, value = model.get_action(obs_tensor)

        next_obs, reward, terminated, truncated, _ = env.step(action.item())

       
        transitions.append({
            "obs": obs,
            "action": action.item(),
            "log_prob": log_prob.item(),
            "value": value.item(),
            "reward": float(reward),
            "terminated": terminated,
            "truncated": truncated,
             # truncated 但没 terminated → 需要存 next_obs 用于 bootstrap
            "next_obs": next_obs if truncated and not terminated else None,
        })

        obs = next_obs
        if terminated or truncated:
            obs, _ = env.reset()

    # rollout 末尾 bootstrap：如果最后一局没结束，计算 V(s_last)
    if not (terminated or truncated):
        with torch.no_grad():
            _, _, bootstrap_value = model.get_action(torch.FloatTensor(obs))
        last_bootstrap = bootstrap_value.item()
    else:
        last_bootstrap = 0.0

    return transitions, last_bootstrap

PPO当中最主要的还是广泛优势估计（GAE） 和 单步优势(delta)计算
1. delta单步优势：这是走了一步之后，拿实际发生的事情和之前的预测做对比算出来的误差（Surprise / Error）。
2. GAE ： 当前这一步的惊喜 \delta_t + 后面所有步的惊喜总和 (打个折扣)

1. 正常结束：

$$\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$$
$$GAE_t = \delta_t + \gamma \lambda GAE_{t+1}$$

对应代码

delta = rewards[step] + gamma * next_value - values[step]

gae = delta + gamma * lam * gae

2. 真正结束（terminated）：
游戏彻底结束，不存任何后续状态。因此，下一个状态的真实价值等于 0（$V(s_{t+1}) = 0$）。同时，当前的死亡与未来的动作没有任何因果关系，不应传播未来的 GAE。

$$\delta_t = r_t - V(s_t)$$
$$GAE_t = \delta_t$$

3. 被截断（Truncated）：
如果给智能体更多时间，它本可以继续玩下去。为了正确估计当前步的价值，我们需要使用神经网络去“预言/估计”如果继续玩下去，下一个新状态的价值（即 Bootstrap 引导），从而补偿由于时间截断造成的损失。但是，因为当前这条采集的轨迹确实在这里断开了，后续的数据和当前轨迹没有因果连续性，所以不能传播未来的 GAE。

$$\delta_t = r_t + \gamma V(s_{next\_obs}) - V(s_t)$$
$$GAE_t = \delta_t$$

这里advantages 是 从当第i步开始算出来的GAE， 然后returns = advantages[i] + values[i] 代表第i步骤开始后续所有基于value算出来的优势差值（GAE） + value 就是真实的优势结果

In [ ]:
# ==========================================
# 第三部分：计算 GAE 优势
# ==========================================
def compute_gae(model, transitions, last_bootstrap, gamma=0.99, lam=0.95):
    """
    广义优势估计，正确处理：
    - terminated（真正结束）：不传播 GAE，V(s')=0
    - truncated（时间截断）：不传播 GAE，但用 V(next_obs) 作为 bootstrap
    - 正常步：正常传播 GAE
    """
    n = len(transitions)
    rewards = [t["reward"] for t in transitions]
    values = [t["value"] for t in transitions]

    # 预计算每个 truncated 步的 bootstrap value
    bootstrap_values = [0.0] * n
    for i, t in enumerate(transitions):
        if t["truncated"] and not t["terminated"] and t["next_obs"] is not None:
            with torch.no_grad():
                _, _, bv = model.get_action(torch.FloatTensor(t["next_obs"]))
            bootstrap_values[i] = bv.item()

    advantages = []
    gae = 0
    next_value = last_bootstrap

    for step in reversed(range(n)):
        t = transitions[step]

        if t["terminated"]:
            # 真正结束：V(s') = 0
            delta = rewards[step] - values[step]
            gae = delta
        elif t["truncated"]:
            # 时间截断：用 V(next_obs) bootstrap，但不传播 GAE
            delta = rewards[step] + gamma * bootstrap_values[step] - values[step]
            gae = delta
        else:
            # 正常步
            delta = rewards[step] + gamma * next_value - values[step]
            gae = delta + gamma * lam * gae

        next_value = values[step]
        advantages.insert(0, gae)

    advantages = torch.FloatTensor(advantages)
    returns = advantages + torch.FloatTensor(values)
    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

    return advantages, returns

In [ ]:
# ==========================================
# 第四部分：PPO 更新
# ==========================================
def ppo_update(model, optimizer, transitions, advantages, returns,
               clip_eps=0.2, epochs=10, batch_size=64):
    """PPO 裁剪目标函数更新"""
    obs = np.array([t["obs"] for t in transitions])
    actions = np.array([t["action"] for t in transitions])
    old_log_probs = np.array([t["log_prob"] for t in transitions])

    obs = torch.FloatTensor(obs)
    actions = torch.LongTensor(actions)
    old_log_probs = torch.FloatTensor(old_log_probs)

    total_policy_loss = 0
    total_value_loss = 0
    total_entropy = 0
    total_kl = 0
    total_clip_frac = 0
    n_updates = 0

    # 这里epoches 轮都使用之前算好的advantage和return
    for _ in range(epochs):
        # 为什么随机
        indices = np.random.permutation(len(transitions))

        for start in range(0, len(transitions), batch_size):
            idx = indices[start:start + batch_size]

            batch_obs = obs[idx]
            batch_actions = actions[idx]
            batch_old_log_probs = old_log_probs[idx]
            batch_advantages = advantages[idx]
            batch_returns = returns[idx]

            logits, values = model(batch_obs)
            dist = torch.distributions.Categorical(logits=logits)
            new_log_probs = dist.log_prob(batch_actions)

            # PPO 裁剪目标
            ratio = torch.exp(new_log_probs - batch_old_log_probs)
            surr1 = ratio * batch_advantages
            surr2 = torch.clamp(ratio, 1 - clip_eps, 1 + clip_eps) * batch_advantages
            policy_loss = -torch.min(surr1, surr2).mean()

            # 价值函数损失，values是critic的预判，batch_return是实际的回报， 这一步相当于让Critic预判的越来越接近实际回报
            value_loss = ((values - batch_returns) ** 2).mean()

            # 熵奖励（鼓励探索）
            entropy = dist.entropy().mean()

            loss = policy_loss + 0.5 * value_loss - 0.0 * entropy

            optimizer.zero_grad()
            loss.backward()
            # 梯度剪裁
            nn.utils.clip_grad_norm_(model.parameters(), 0.5)
            optimizer.step()

            # 统计指标
            with torch.no_grad():
                total_kl += (batch_old_log_probs - new_log_probs).mean().item()
                total_clip_frac += ((ratio - 1.0).abs() > clip_eps).float().mean().item()

            total_policy_loss += policy_loss.item()
            total_value_loss += value_loss.item()
            total_entropy += entropy.item()
            n_updates += 1

    return {
        "policy_loss": total_policy_loss / n_updates,
        "value_loss": total_value_loss / n_updates,
        "entropy": total_entropy / n_updates,
        "approx_kl": total_kl / n_updates,
        "clip_fraction": total_clip_frac / n_updates,
    }


In [ ]:
def train():
    # 1. 初始化环境和模型
    env = gym.make("CartPole-v1")
    model = ActorCritic()
    # 设定优化器，学习率固定为 3e-4，去掉了复杂的学习率衰减
    optimizer = optim.Adam(model.parameters(), lr=3e-4)

    total_iterations = 40     # 总共训练 40 轮
    steps_per_rollout = 2048  # 每轮收集 2048 步数据

    print("开始极简版 PPO 训练...")

    for iteration in range(total_iterations):
        # --------------------------------------------------
        # 步骤 1：与环境交互，收集数据 (Rollout)
        # --------------------------------------------------
        transitions, last_bootstrap = collect_rollout(model, env, steps_per_rollout)

        # 统计本轮收集到的平均得分 (仅用于打印看效果，不参与梯度更新)
        ep_rewards = []
        ep_reward = 0
        for t in transitions:
            ep_reward += t["reward"]
            if t["terminated"] or t["truncated"]:
                ep_rewards.append(ep_reward)
                ep_reward = 0
        mean_reward = np.mean(ep_rewards) if ep_rewards else 0

        # --------------------------------------------------
        # 步骤 2：计算优势 (Advantages) 和 目标回报 (Returns)
        # --------------------------------------------------
        advantages, returns = compute_gae(model, transitions, last_bootstrap)

        # --------------------------------------------------
        # 步骤 3：执行 PPO 核心损失计算与网络更新
        # --------------------------------------------------
        metrics = ppo_update(model, optimizer, transitions, advantages, returns)

        # --------------------------------------------------
        # 步骤 4：打印当前进度
        # --------------------------------------------------
        print(f"迭代 {iteration + 1:2d}/{total_iterations} | "
              f"平均得分: {mean_reward:6.1f} | "
              f"策略损失(Actor): {metrics['policy_loss']:.4f} | "
              f"价值损失(Critic): {metrics['value_loss']:.4f}")

    print("训练完成！")
    env.close()

if __name__ == "__main__":
    train()